### 学习心得

garbage in garbage out 数据加载很关键。文档加载器负责将非结构化数据统一为可操作的结构化数据，要成适合检索建模的语料，不仅是纯文本内容，还有其他关键信息（像来源页码作者）方便溯源。

文本分块技术将加载后的长篇文档，切分成更小、更易于处理的单元。Embedding model嵌入模型有严格的上下文token限制，该类模型把人类语言（文本、图片等）翻译成计算机能直接计算的向量（有点像bert？编码器）

文本分块的大小并非越大越好，现代大语言模型倾向于两头而忽略中间，大块文本自带的池化过程（求token向量平均）会稀释真正有用的信息，存在噪声。分块策略有点像数据结构里学到的树结构。

LlamaIndex中叫节点是原始文档被加载、解析和切分后形成的基本数据单元，他是结构化类似自带指针能带点上下文还知道位置信息。

In [1]:
# 1. 导入所需模块
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

# 2. 加载文档（蜂医.txt 与 day2.ipynb 同目录）
loader = TextLoader("蜂医.txt", encoding="utf-8")
docs = loader.load()

# 3. 策略一：CharacterTextSplitter（固定单分隔符切分，默认按 "\n\n"）
char_splitter = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=10
)
char_chunks = char_splitter.split_documents(docs)

print(f"【CharacterTextSplitter】文本被切分为 {len(char_chunks)} 个块。\n")
print("--- 前5个块内容示例 ---")
for i, chunk in enumerate(char_chunks[:5]):
    print("=" * 60)
    print(f'块 {i+1} (长度: {len(chunk.page_content)}): "{chunk.page_content}"')

print("\n" + "#" * 60 + "\n")

# 4. 策略二：RecursiveCharacterTextSplitter（递归多级分隔符切分）
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", "。", "，", " ", ""],
    chunk_size=200,
    chunk_overlap=10,
)
recursive_chunks = recursive_splitter.split_documents(docs)

print(f"【RecursiveCharacterTextSplitter】文本被切分为 {len(recursive_chunks)} 个块。\n")
print("--- 前5个块内容示例 ---")
for i, chunk in enumerate(recursive_chunks[:5]):
    print("=" * 60)
    print(f'块 {i+1} (长度: {len(chunk.page_content)}): "{chunk.page_content}"')

C:\Users\35368\AppData\Local\Temp\ipykernel_21484\843629796.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
Created a chunk of size 201, which is longer than the specified 200
Created a chunk of size 277, which is longer than the specified 200
Created a chunk of size 296, which is longer than the specified 200


【CharacterTextSplitter】文本被切分为 14 个块。

--- 前5个块内容示例 ---
块 1 (长度: 72): "# 蜂医

游戏《三角洲行动》中的支援型干员

蜂医是2024年琳琅天上发行的《三角洲行动》中的支援型干员之一，在早期版本是唯一一个支援型干员。"
块 2 (长度: 201): "蜂医在游戏中能够使用战术装备“激素枪”：远程治疗队友或'自我治疗'，还可以使用兵种道具“烟幕无人机”：释放长烟幕，和“蜂巢科技烟雾弹”：产生一团白色烟雾（使用激素枪对烟雾射击换变成绿色烟雾，可起到治疗作用），干员特长为“高效救援”：救援倒地队友时速度更快，在全面战场模式中约1.4秒就能救起队友，且被救起的队友能恢复更多生命值。在烽火地带中，还能够移除队友血量上限减少的负面效果。 \[1-2]****"
块 3 (长度: 189): "* 中文名

  罗伊•斯米

* 外文名

  Roy smee \[2]**

* 别    名

  罗伊、蜂医

* 性    别

  男

- 登场作品

  [三角洲行动](/item/%E4%B8%89%E8%A7%92%E6%B4%B2%E8%A1%8C%E5%8A%A8/63251542?fromModule=lemma_inlink)

- 生    日"
块 4 (长度: 133): "- 生    日

  2008年2月23日 \[3]**

- 身    高

  176 cm \[3]**

- 体    重

  75 kg \[3]**

## 目录

1. 1[角色设定](#1)
2. 2[角色定位](#2)
3. 3[技能](#3)"
块 5 (长度: 189): "1) ▪[战术装备 - 激素枪](#3-1)
2) ▪[战术道具 - 烟幕无人机](#3-2)
3) ▪[战术道具 - 蜂巢科技烟雾弹](#3-3)

1. ▪[干员特长 - 高效救援](#3-4)

## 角色设定

三角洲行动：医疗角色蜂医！让你不再为血量担忧

蜂医是游戏中的一名战地医生，拥有丰富的参军履历。

蜂医在干员档案中标明他有一个妻子和女儿。

## 角色定位"

############################################################

【Recursiv

### 两种文本分块策略对比：`CharacterTextSplitter` vs `RecursiveCharacterTextSplitter`

| 对比维度 | CharacterTextSplitter（固定分隔符切分） | RecursiveCharacterTextSplitter（递归字符切分） |
| :--- | :--- | :--- |
| **切分逻辑** | 仅按**单一分隔符**（默认为 `\n\n`）切分后再合并，逻辑简单直接。 | 按**优先级列表**（如 `\n\n` → `\n` → `。` → `，` → 空格 → `""`）逐层递归切分，类似树结构的逐级下钻。 |
| **长度控制** | 较弱。若单个段落本身超过 `chunk_size`，无法继续细切，会导致块严重超长并触发警告。 | 较强。大段落超长时会自动降级用下一级分隔符继续切分，能严格贴近 `chunk_size` 限制。 |
| **语义完整性** | 依赖原文排版；若按单字符或不当分隔符切分，容易在句子中间硬截断。 | 较好。优先保留完整段落，其次保留完整句子和短语，最大程度避免破坏上下文语义。 |
| **中文适配度** | 一般，默认仅识别双换行，面对长段落中文文档时效果不理想。 | 优秀，通过在 `separators` 中加入 `"。"`、`"，"` 等中文标点即可自然断句。 |
| **适用场景** | 结构极其规整、段落长度均匀且有固定分隔标记的文本（如日志、格式化清单）。 | **绝大多数通用 RAG 场景的默认首选**，尤其适合长文章、百科、说明书等自然语言文档。 |

> **小结**：`CharacterTextSplitter` 是一刀切，容易出现“切不动导致超长”或“切太碎破坏语义”的问题；`RecursiveCharacterTextSplitter` 通过“先粗后细、层层降级”的递归策略，在**控制块大小**与**保持语义连贯**之间取得了更好的平衡。

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"当前使用设备: {device}")

# 1. 加载本地中文 Embedding 模型
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-zh-v1.5",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)

# 2. 初始化 SemanticChunker
text_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    sentence_split_regex=r"(?<=[。！？.!?\n])\s*"
)

# 3. 加载文档并执行语义切分
loader = TextLoader("蜂医.txt", encoding="utf-8")
documents = loader.load()

docs = text_splitter.split_documents(documents)

# 4. 查看切分结果
print(f"【SemanticChunker】文本被切分为 {len(docs)} 个块。\n")
print("--- 前5个块内容示例 ---")
for i, chunk in enumerate(docs[:5]):
    print("=" * 60)
    print(f'块 {i+1} (长度: {len(chunk.page_content)}): "{chunk.page_content}"')

C:\Users\35368\AppData\Local\Temp\ipykernel_3628\2877759811.py:5: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


当前使用设备: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\miniconda\envs\pytorch\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\35368\.cache\huggingface\hub\models--BAAI--bge-small-zh-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/27.7k [00:00<?, ?B/s]

... (8 outputs left out)

tokenizer.json:   0%|          | 0.00/439k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

【SemanticChunker】文本被切分为 7 个块。

--- 前5个块内容示例 ---
块 1 (长度: 280): "# 蜂医
  游戏《三角洲行动》中的支援型干员
  蜂医是2024年琳琅天上发行的《三角洲行动》中的支援型干员之一，在早期版本是唯一一个支援型干员。  蜂医在游戏中能够使用战术装备“激素枪”：远程治疗队友或'自我治疗'，还可以使用兵种道具“烟幕无人机”：释放长烟幕，和“蜂巢科技烟雾弹”：产生一团白色烟雾（使用激素枪对烟雾射击换变成绿色烟雾，可起到治疗作用），干员特长为“高效救援”：救援倒地队友时速度更快，在全面战场模式中约1. 4秒就能救起队友，且被救起的队友能恢复更多生命值。 在烽火地带中，还能够移除队友血量上限减少的负面效果。 \[1-2]****
"
块 2 (长度: 176): " * 中文名
 罗伊•斯米
  * 外文名
 Roy smee \[2]**
  * 别    名
 罗伊、蜂医
  * 性    别
 男
  - 登场作品
 [三角洲行动](/item/%E4%B8%89%E8%A7%92%E6%B4%B2%E8%A1%8C%E5%8A%A8/63251542? fromModule=lemma_inlink)
"
块 3 (长度: 441): " - 生    日
 2008年2月23日 \[3]**
  - 身    高
 176 cm \[3]**
  - 体    重
 75 kg \[3]**
  ## 目录
  1. 1[角色设定](#1)
 2. 2[角色定位](#2)
 3. 3[技能](#3)
  1) ▪[战术装备 - 激素枪](#3-1)
 2) ▪[战术道具 - 烟幕无人机](#3-2)
 3) ▪[战术道具 - 蜂巢科技烟雾弹](#3-3)
  1. ▪[干员特长 - 高效救援](#3-4)
  ## 角色设定
  三角洲行动：医疗角色蜂医！ 让你不再为血量担忧
  蜂医是游戏中的一名战地医生，拥有丰富的参军履历。  蜂医在干员档案中标明他有一个妻子和女儿。  ## 角色定位
  -蜂医在团队中主要承担支援和治疗的职责，是团队生存能力的重要保障。 他能在激烈的战斗中为队友提供及时的治疗，确保队友保持良好的战斗状态，还能利用烟幕技能为团队创造有利的战术环境，如掩护队友转移、推进或突袭等。 "
块

### 课程反馈

这章着重教学了数据加载和文本分块，主要是面向rag系统下的外部数据库，个人感觉还是挺好上手的而且了解到了他的核心运行逻辑，是为了解决什么问题，当下又有哪些像文档加载器和文本分块的形式。

### task2

In [1]:
import time
from collections import Counter
from unstructured.partition.auto import partition

pdf_path = "rag.pdf"

start = time.time()
elements = partition(
    filename=pdf_path,
    content_type="application/pdf"
)
print(f"耗时: {time.time() - start:.2f} 秒")
print(f"解析完成: {len(elements)} 个元素, {sum(len(str(e)) for e in elements)} 字符")

types = Counter(e.category for e in elements)
print(f"元素类型: {dict(types)}")

print("\n前 5 个元素预览:")
for i, element in enumerate(elements[:5], 1):
    print(f"Element {i} ({element.category}):")
    print(element)
    print("=" * 60)

Thank you for using Unstructured Open Source! Try Unstructured's commercial offering for improved output quality. 10,000 free pages to start. Learn more: https://transform.unstructured.io/?utm_source=unstructured_oss Hide this notice: UNSTRUCTURED_DISABLE_NOTICE=1
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor bec

耗时: 5.16 秒
解析完成: 279 个元素, 7500 字符
元素类型: {'Header': 22, 'Title': 184, 'NarrativeText': 14, 'UncategorizedText': 41, 'Footer': 15, 'ListItem': 3}

前 5 个元素预览:
Element 1 (Header):
网页
Element 2 (Header):
新闻
Element 3 (Header):
贴吧
Element 4 (Header):
知道
Element 5 (Header):
网盘


In [2]:
import os, shutil

env_root = r"d:\miniconda\envs\pytorch"
env_bin = os.path.join(env_root, "Library", "bin")
tess_exe = os.path.join(env_bin, "tesseract.exe")

print("1. conda 里有 tesseract.exe 吗:", os.path.exists(tess_exe))
print("2. 当前 PATH 能找到 tesseract 吗:", shutil.which("tesseract"))

# 如果其实已经装在 conda 里，直接补进当前会话的 PATH
if os.path.exists(tess_exe):
    os.environ["PATH"] = env_bin + ";" + os.environ["PATH"]
    # 顺便找一下语言包目录
    for candidate in [
        os.path.join(env_root, "share", "tessdata"),
        os.path.join(env_root, "Library", "share", "tessdata"),
    ]:
        if os.path.exists(candidate):
            os.environ["TESSDATA_PREFIX"] = candidate
            print("3. 找到语言包目录:", candidate)
            print("   包含 chi_sim 吗:", os.path.exists(os.path.join(candidate, "chi_sim.traineddata")))
            break
    print("4. 补完 PATH 后能找到吗:", shutil.which("tesseract"))

1. conda 里有 tesseract.exe 吗: True
2. 当前 PATH 能找到 tesseract 吗: None
3. 找到语言包目录: d:\miniconda\envs\pytorch\share\tessdata
   包含 chi_sim 吗: True
4. 补完 PATH 后能找到吗: d:\miniconda\envs\pytorch\Library\bin\tesseract.EXE


In [3]:
import time
import logging
from collections import Counter
from unstructured.partition.pdf import partition_pdf

# 屏蔽 pdfminer 刷屏警告
logging.getLogger("pdfminer").setLevel(logging.ERROR)

pdf_path = "rag.pdf"

start = time.time()
elements_hires = partition_pdf(
    filename=pdf_path,
    strategy="hi_res",
    languages=["chi_sim", "eng"],
    infer_table_structure=True,
)
print(f"[hi_res] 耗时: {time.time() - start:.2f} 秒")
print(f"[hi_res] 解析完成: {len(elements_hires)} 个元素, {sum(len(str(e)) for e in elements_hires)} 字符")
print(f"[hi_res] 元素类型: {dict(Counter(e.category for e in elements_hires))}")

print("\n前 5 个元素预览:")
for i, element in enumerate(elements_hires[:5], 1):
    print(f"Element {i} ({element.category}):")
    print(element)
    print("=" * 60)

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

[hi_res] 耗时: 32.06 秒
[hi_res] 解析完成: 243 个元素, 8385 字符
[hi_res] 元素类型: {'Image': 21, 'UncategorizedText': 114, 'Header': 3, 'NarrativeText': 63, 'Title': 31, 'Table': 4, 'FigureCaption': 4, 'ListItem': 3}

前 5 个元素预览:
Element 1 (Image):

Element 2 (UncategorizedText):
页
Element 3 (UncategorizedText):
网页
Element 4 (UncategorizedText):
闻
Element 5 (UncategorizedText):
新闻


In [4]:
import time
from collections import Counter
from unstructured.partition.pdf import partition_pdf

pdf_path = "rag.pdf"

start = time.time()
elements_ocr = partition_pdf(
    filename=pdf_path,
    strategy="ocr_only",
    languages=["chi_sim", "eng"],
)
print(f"[ocr_only] 耗时: {time.time() - start:.2f} 秒")
print(f"[ocr_only] 解析完成: {len(elements_ocr)} 个元素, {sum(len(str(e)) for e in elements_ocr)} 字符")
print(f"[ocr_only] 元素类型: {dict(Counter(e.category for e in elements_ocr))}")

print("\n前 5 个元素预览:")
for i, element in enumerate(elements_ocr[:5], 1):
    print(f"Element {i} ({element.category}):")
    print(element)
    print("=" * 60)

[ocr_only] 耗时: 34.70 秒
[ocr_only] 解析完成: 103 个元素, 2514 字符
[ocr_only] 元素类型: {'Title': 50, 'NarrativeText': 14, 'UncategorizedText': 39}

前 5 个元素预览:
Element 1 (Title):
知道 视频 文库 资讯 采购 百科 SR ”注册
Element 2 (NarrativeText):
ee Bai ®t! a tae | | sy
Element 3 (UncategorizedText):
!
Element 4 (NarrativeText):
首页 Pear
Element 5 (Title):
生成 人 Za 增 播报 Owe | GO


### `partition` 与 `partition_pdf`（`hi_res` / `ocr_only`）解析对比

将通用的 `partition` 替换为专属的 `partition_pdf` 后，可以显式指定解析策略（`strategy`）、中英文语言包（`languages=["chi_sim", "eng"]`）及表格结构识别（`infer_table_structure=True`）。基于 `rag.pdf`（自带文字层的网页导出 PDF）的实测结果对比如下：

| 解析方式 | 耗时 | 元素数 / 字符数 | 核心结构识别 | 输出质量观察 |
| :--- | :--- | :--- | :--- | :--- |
| `partition`（默认 `auto`/`fast`） | **5.16s** | 279 / 7,500 | 无表格、无图片（`Title` 184 / `NarrativeText` 14） | 速度最快、文字提取准确，但缺乏版面理解，大量正文被误判为标题 |
| `partition_pdf`（`hi_res`） | 32.06s | 243 / **8,385** | 识别出 **4 个表格、21 张图片、4 个图注**（`Title` 31 / `NarrativeText` 63） | 结构还原度最高、召回字符最多，但耗时增加约 6 倍，且导航栏等密集区域会出现重复碎片（如“页 / 网页”） |
| `partition_pdf`（`ocr_only`） | 34.70s | 103 / 2,514 | 无表格、无图片、无页眉页脚 | **效果最差**：忽略了原电子文本层，耗时最长，字符流失约 70%，且出现大量乱码（如 `ee Bai ®t! a tae`） |

#### 核心变化与结论
1. **默认 `partition`（`fast` 路径）**：直接读取 PDF 底层文字流，**快且字准，但“不懂排版”**，会丢失表格和图片信息，适合纯文字电子文档。
2. **`hi_res` 策略**：引入视觉版面分析与表格模型，**大幅提升文档结构还原能力**（准确区分正文/标题，保留表格与图文），最适合复杂排版 PDF 的 RAG 入库，只需在后处理阶段过滤少量未分类碎片（`UncategorizedText`）。
3. **`ocr_only` 策略**：强制转图后走 Tesseract OCR，对本身带文字层的电子 PDF 属于“舍近求远”，易受网页图标与中英混排干扰导致严重丢字和乱码，**仅适用于无文字层的纯扫描件 PDF**。